# Notebook 08: The Alignment Zoo -- KTO, IPO, SimPO, ORPO

**Sprint 2 | Alignment Track**

---

**What you will learn**:
- Four major alternatives to DPO, each addressing a specific limitation
- From-scratch implementations of KTO, IPO, SimPO, and ORPO losses
- When to use which method -- a practical decision framework
- The trend in alignment: simpler methods, fewer models, less memory

**Prerequisites**: Notebook 07 (DPO from RLHF). You must understand DPO before studying its variants.

**Runtime**: Google Colab with T4 GPU (free tier).

---
## 1. Self-Quiz: Active Recall Before Learning

Answer these without looking anything up:

1. **Name 4 alternatives to DPO** for preference-based alignment.
2. **Which method doesn't need paired preferences?** (i.e., you just need "good" and "bad" examples, not side-by-side comparisons)
3. **Which method removes the reference model entirely?** What replaces it?
4. **Which method combines SFT and alignment into one stage?** Why is this useful?
5. **What is the main limitation of DPO that all these methods try to address?**

---
*Come back to these after completing the notebook. You should nail all five.*

---
## 2. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We'll implement all four losses and test them on synthetic data first,
# then compare their behavior side-by-side.

# Recap: the DPO loss for reference
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """Standard DPO loss (from Notebook 07)."""
    chosen_logratios = policy_chosen_logps - ref_chosen_logps
    rejected_logratios = policy_rejected_logps - ref_rejected_logps
    logits = beta * (chosen_logratios - rejected_logratios)
    loss = -F.logsigmoid(logits).mean()
    return loss

print("Setup complete. DPO baseline loss loaded for comparison.")

---
## 3. Why So Many Methods?

DPO was a breakthrough, but it has specific limitations that motivated a wave of follow-up work. Each variant in this notebook addresses a different weakness:

| Method | Year | Key Innovation | Addresses DPO Limitation |
|--------|------|---------------|-------------------------|
| **KTO** | 2024 | Prospect theory loss; works with unpaired data | Paired preference data is expensive to collect |
| **IPO** | 2023 | Regularized loss to prevent overfitting | DPO can overfit due to deterministic reward-policy mapping |
| **SimPO** | 2024 | Removes reference model; length-normalized rewards | Reference model doubles memory; length exploitation |
| **ORPO** | 2024 | Combines SFT + alignment in one stage | Two-stage pipeline (SFT then DPO) is wasteful |

### The Progression

```
RLHF (PPO)          4 models, 3 phases, complex
  |                      |
  v                      v
DPO                  2 models, 2 phases, simple
  |                      |
  +-- KTO             2 models, 2 phases, unpaired data
  +-- IPO             2 models, 2 phases, more robust
  +-- SimPO           1 model,  2 phases, no reference
  +-- ORPO            1 model,  1 phase,  SFT + alignment together
```

The trend is clear: **fewer models, fewer phases, simpler data requirements**.

---
## 4. KTO: Kahneman-Tversky Optimization

**Paper**: Ethayarajh et al. (2024), "KTO: Model Alignment as Prospect Theoretic Optimization"  
**Link**: [https://arxiv.org/abs/2402.01306](https://arxiv.org/abs/2402.01306)

### The Problem KTO Solves

DPO requires **paired** preferences: for each prompt, you need both a chosen AND a rejected response. This is expensive to collect because annotators must compare two outputs side-by-side.

But in practice, we often have **unpaired** feedback:
- Thumbs up / thumbs down on individual responses
- User satisfaction ratings (without a comparison)
- Safety labels ("this response is harmful" vs. "this is fine")

### The Key Insight: Prospect Theory

KTO draws on Kahneman & Tversky's prospect theory from behavioral economics:
- **Loss aversion**: People feel losses more strongly than equivalent gains
- **Reference dependence**: Utility is defined relative to a reference point

Applied to alignment:
- **Desirable outputs** (y is good): We want the policy to assign higher probability than the reference, but the utility saturates (diminishing returns)
- **Undesirable outputs** (y is bad): We want the policy to assign lower probability than the reference, and the penalty is steeper (loss aversion)

### The Math

Define the implicit reward $r_\theta(x, y) = \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$ and a reference point $z_0$, which is an estimate of $\text{KL}(\pi_\theta \| \pi_{\text{ref}})$ over the data (note: there is no $\beta$ inside $z_0$; $\beta$ multiplies the whole deviation from the reference point).

For a desirable output $y_d$ (good):
$$\mathcal{L}_{\text{KTO}}^{\text{desirable}} = 1 - \sigma\left(\beta \left(r_\theta(x, y_d) - z_0\right)\right)$$

For an undesirable output $y_u$ (bad):
$$\mathcal{L}_{\text{KTO}}^{\text{undesirable}} = 1 - \sigma\left(\beta \left(z_0 - r_\theta(x, y_u)\right)\right)$$

Note: the paper (Ethayarajh et al. 2024) estimates $z_0$ from mismatched pairs (responses matched with other prompts in the batch) with a $\max(0, \cdot)$ clamp; the batch-mean version used in this notebook's code is a simplification (the code is self-consistent).

The combined loss weights the two classes with $\lambda_D$ and $\lambda_U$. These weights handle **class imbalance**, not loss aversion: the paper's guidance is to set $\frac{\lambda_D n_D}{\lambda_U n_U} \in \left[1, \frac{4}{3}\right]$ (where $n_D, n_U$ are the desirable/undesirable example counts), which if anything weights the desirable side $\geq$ the undesirable side. Loss aversion in KTO comes from the shape of the value function itself, not from $\lambda_U > \lambda_D$:
$$\mathcal{L}_{\text{KTO}} = \lambda_D \cdot \mathcal{L}^{\text{desirable}} + \lambda_U \cdot \mathcal{L}^{\text{undesirable}}$$

In [ ]:
def kto_loss(
    policy_logps: torch.Tensor,
    ref_logps: torch.Tensor,
    is_desirable: torch.Tensor,
    beta: float = 0.1,
    desirable_weight: float = 1.0,
    undesirable_weight: float = 1.0,
) -> tuple[torch.Tensor, dict]:
    """
    Compute the KTO (Kahneman-Tversky Optimization) loss.
    
    Key difference from DPO: operates on INDIVIDUAL examples, not pairs.
    Each example is labeled as desirable (good) or undesirable (bad).
    
    Args:
        policy_logps: Log probs under policy. Shape: (batch_size,)
        ref_logps: Log probs under reference. Shape: (batch_size,)
        is_desirable: Boolean mask. True = desirable (good), False = undesirable (bad).
                      Shape: (batch_size,)
        beta: Temperature parameter (same role as in DPO).
        desirable_weight: Weight for desirable loss (lambda_D).
        undesirable_weight: Weight for undesirable loss (lambda_U).
    
    Returns:
        loss: Scalar KTO loss.
        metrics: Dict with per-component losses and KL estimate.
    """
    # Log-ratio: how much policy deviates from reference
    logratios = policy_logps - ref_logps  # (batch_size,)
    
    # Estimate z_ref: the average KL divergence across the batch
    # This is the "reference point" in prospect theory
    # KL(pi || pi_ref) approx= E[log(pi/pi_ref)] for samples from pi
    # We use the batch mean as an estimate
    z_ref = beta * logratios.detach().mean()
    
    # Separate desirable and undesirable examples
    desirable_mask = is_desirable.bool()
    undesirable_mask = ~desirable_mask
    
    # Desirable loss: we want log-ratio to be HIGH (above z_ref)
    # Loss = 1 - sigma(beta * (logratio - z_ref))
    # When logratio > z_ref/beta, sigma is large, loss is small (good)
    desirable_logratios = logratios[desirable_mask]
    if len(desirable_logratios) > 0:
        desirable_loss = (1.0 - torch.sigmoid(beta * desirable_logratios - z_ref)).mean()
    else:
        desirable_loss = torch.tensor(0.0, device=policy_logps.device)
    
    # Undesirable loss: we want log-ratio to be LOW (below z_ref)
    # Loss = 1 - sigma(beta * (z_ref - logratio))
    # When logratio < z_ref/beta, sigma is large, loss is small (good)
    undesirable_logratios = logratios[undesirable_mask]
    if len(undesirable_logratios) > 0:
        undesirable_loss = (1.0 - torch.sigmoid(z_ref - beta * undesirable_logratios)).mean()
    else:
        undesirable_loss = torch.tensor(0.0, device=policy_logps.device)
    
    # Combined loss with asymmetric weighting (loss aversion)
    loss = desirable_weight * desirable_loss + undesirable_weight * undesirable_loss
    
    metrics = {
        'desirable_loss': desirable_loss.item(),
        'undesirable_loss': undesirable_loss.item(),
        'z_ref': z_ref.item(),
        'mean_logratio_desirable': desirable_logratios.mean().item() if len(desirable_logratios) > 0 else 0.0,
        'mean_logratio_undesirable': undesirable_logratios.mean().item() if len(undesirable_logratios) > 0 else 0.0,
    }
    
    return loss, metrics


print("KTO loss function defined.")

In [ ]:
# Test KTO loss with synthetic data

print("=" * 60)
print("TEST: KTO on synthetic data")
print("=" * 60)

# Scenario: 4 examples, 2 desirable, 2 undesirable
# Policy assigns higher prob to desirable, lower to undesirable (correct behavior)
policy_logps = torch.tensor([-1.0, -1.5, -4.0, -5.0])  # first 2: desirable, last 2: undesirable
ref_logps = torch.tensor([-2.0, -2.0, -2.0, -2.0])     # reference is neutral
is_desirable = torch.tensor([True, True, False, False])

loss, metrics = kto_loss(policy_logps, ref_logps, is_desirable, beta=0.1)
print(f"\nScenario: Policy correctly upweights desirable, downweights undesirable")
print(f"Loss: {loss.item():.4f} (should be LOW)")
print(f"Metrics: {metrics}")

# Scenario: Policy is wrong (upweights undesirable, downweights desirable)
policy_logps_wrong = torch.tensor([-4.0, -5.0, -1.0, -1.5])  # reversed!
loss_wrong, metrics_wrong = kto_loss(policy_logps_wrong, ref_logps, is_desirable, beta=0.1)
print(f"\nScenario: Policy incorrectly upweights undesirable, downweights desirable")
print(f"Loss: {loss_wrong.item():.4f} (should be HIGH)")
print(f"Metrics: {metrics_wrong}")

# Test asymmetric weighting (loss aversion)
print("\n" + "=" * 60)
print("Effect of asymmetric weighting (loss aversion)")
print("=" * 60)

for lam_u in [0.5, 1.0, 2.0, 5.0]:
    l, _ = kto_loss(policy_logps, ref_logps, is_desirable, beta=0.1,
                    desirable_weight=1.0, undesirable_weight=lam_u)
    print(f"lambda_U={lam_u:.1f} -> loss={l.item():.4f} (higher weight on avoiding bad outputs)")

### Why KTO Matters

1. **Data efficiency**: You don't need to pair responses. Any dataset with binary quality labels works.
2. **Scalability**: Collecting thumbs-up/thumbs-down is much cheaper than collecting pairwise comparisons.
3. **Natural fit**: Most real-world feedback is unpaired (user ratings, safety labels, helpfulness flags).
4. **Performance**: Surprisingly, KTO matches DPO performance in many benchmarks despite using weaker supervision.

### When KTO Breaks

- When the quality of binary labels is low (noisy "good"/"bad" labels)
- When the distribution of desirable vs. undesirable examples is very imbalanced
- The z_ref estimate can be noisy with small batch sizes

---
## 5. IPO: Identity Preference Optimization

**Paper**: Azar et al. (2023), "A General Theoretical Paradigm to Understand Learning from Human Feedback"  
**Link**: [https://arxiv.org/abs/2310.12036](https://arxiv.org/abs/2310.12036)

### The Problem IPO Solves

DPO has a subtle overfitting problem. The DPO loss uses the **log-sigmoid**, which can saturate:

- When the policy strongly prefers chosen over rejected, $\sigma(\text{logit}) \to 1$, and the loss $-\log\sigma(\text{logit}) \to 0$
- This means the gradient vanishes, and the policy stops getting signal
- But the policy may continue to push the log-ratio further apart ("overfitting" to the preference margin)

This is related to the **deterministic** mapping between rewards and policies in DPO. The Bradley-Terry model assumes a specific functional form, and if the true preferences don't follow this form exactly, DPO can overfit.

### The Key Insight: Regularize with a Target Margin

Instead of pushing the log-ratio difference through a sigmoid (which saturates), IPO uses a **squared loss** that penalizes deviation from a target margin:

$$\mathcal{L}_{\text{IPO}} = \left(\left(\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right) - \frac{1}{2\tau}\right)^2$$

where $\tau$ plays the role of $\beta$ itself -- the KL-regularization strength parameter; the squared-loss **target margin** is $1/(2\tau)$.

The key difference: the squared loss **never saturates**. Even when the margin is correct, the loss provides gradient signal to prevent it from growing further.

In [ ]:
def ipo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    tau: float = 0.5,
) -> tuple[torch.Tensor, dict]:
    """
    Compute the IPO (Identity Preference Optimization) loss.
    
    Key difference from DPO: uses squared loss instead of log-sigmoid,
    which prevents overfitting by never saturating.
    
    Args:
        policy_chosen_logps: Log probs of chosen under policy. Shape: (batch_size,)
        policy_rejected_logps: Log probs of rejected under policy. Shape: (batch_size,)
        ref_chosen_logps: Log probs of chosen under reference. Shape: (batch_size,)
        ref_rejected_logps: Log probs of rejected under reference. Shape: (batch_size,)
        tau: Regularization parameter. Controls the target margin.
             Larger tau = smaller target margin = more regularization.
    
    Returns:
        loss: Scalar IPO loss.
        metrics: Dict with margin statistics.
    """
    # Log-ratios
    chosen_logratios = policy_chosen_logps - ref_chosen_logps
    rejected_logratios = policy_rejected_logps - ref_rejected_logps
    
    # Margin: how much policy prefers chosen over rejected (relative to reference)
    margin = chosen_logratios - rejected_logratios  # (batch_size,)
    
    # Target margin: 1/(2*tau)
    target_margin = 1.0 / (2.0 * tau)
    
    # IPO loss: squared deviation from target margin
    loss = ((margin - target_margin) ** 2).mean()
    
    metrics = {
        'mean_margin': margin.mean().item(),
        'target_margin': target_margin,
        'margin_deviation': (margin - target_margin).abs().mean().item(),
    }
    
    return loss, metrics


print("IPO loss function defined.")

In [ ]:
# Test IPO loss and compare saturation behavior with DPO

print("=" * 60)
print("Comparing DPO vs IPO: saturation behavior")
print("=" * 60)

# Sweep the margin from strongly-wrong to strongly-right
ref_chosen = torch.tensor([-3.0])
ref_rejected = torch.tensor([-3.0])

margins = np.linspace(-5, 10, 100)
dpo_losses = []
ipo_losses = []
dpo_grads = []
ipo_grads = []

for m in margins:
    # Create policy log-probs with this margin
    pc = torch.tensor([ref_chosen.item() + m / 2], requires_grad=True)
    pr = torch.tensor([ref_rejected.item() - m / 2], requires_grad=True)
    
    # DPO loss
    dl = dpo_loss(pc, pr, ref_chosen, ref_rejected, beta=0.1)
    dl.backward()
    dpo_losses.append(dl.item())
    dpo_grads.append(pc.grad.item())
    
    # IPO loss
    pc2 = torch.tensor([ref_chosen.item() + m / 2], requires_grad=True)
    pr2 = torch.tensor([ref_rejected.item() - m / 2], requires_grad=True)
    il, _ = ipo_loss(pc2, pr2, ref_chosen, ref_rejected, tau=0.5)
    il.backward()
    ipo_losses.append(il.item())
    ipo_grads.append(pc2.grad.item())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(margins, dpo_losses, label='DPO', linewidth=2)
axes[0].plot(margins, ipo_losses, label='IPO', linewidth=2)
axes[0].set_xlabel('Log-ratio margin (chosen - rejected)', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Loss vs Margin', fontsize=14)
axes[0].legend(fontsize=12)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.3)

axes[1].plot(margins, dpo_grads, label='DPO gradient', linewidth=2)
axes[1].plot(margins, ipo_grads, label='IPO gradient', linewidth=2)
axes[1].set_xlabel('Log-ratio margin (chosen - rejected)', fontsize=12)
axes[1].set_ylabel('Gradient (w.r.t. chosen logps)', fontsize=12)
axes[1].set_title('Gradient vs Margin', fontsize=14)
axes[1].legend(fontsize=12)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observation:")
print("- DPO gradient vanishes as margin increases (sigmoid saturates)")
print("- IPO gradient remains non-zero and REVERSES past the target margin")
print("- IPO actively pulls the margin back toward the target, preventing overfitting")

### Why IPO Matters

1. **Prevents overfitting**: The squared loss never saturates, so the model always gets gradient signal.
2. **Target margin**: Instead of "make chosen better than rejected" (unbounded), IPO says "make chosen better by exactly this much" (bounded).
3. **Theoretical grounding**: IPO comes from a general framework that unifies several alignment methods.

### When IPO Breaks

- The target margin $1/(2\tau)$ may not be appropriate for all preference pairs (some are clear, others are ambiguous)
- The squared loss can be sensitive to outliers (large margins get quadratic penalty)
- Still requires paired preferences (unlike KTO)

---
## 6. SimPO: Simple Preference Optimization

**Paper**: Meng et al. (2024), "SimPO: Simple Preference Optimization with a Reference-Free Reward"  
**Link**: [https://arxiv.org/abs/2405.14734](https://arxiv.org/abs/2405.14734)

### The Problem SimPO Solves

DPO requires a **reference model** in memory, which:
- Doubles GPU memory requirements
- Requires careful synchronization
- Adds implementation complexity

Additionally, DPO's implicit reward uses **unnormalized** log-probabilities that grow with response length, creating a length exploitation vulnerability.

### The Key Insight: Length-Normalized Log-Likelihood as Reward

SimPO replaces the log-ratio $\log \pi/\pi_{\text{ref}}$ with the **average log-likelihood** of the response:

$$r_{\text{SimPO}}(x, y) = \frac{1}{|y|} \log \pi_\theta(y|x)$$

This has two important properties:
1. **No reference model needed**: The reward is just the policy's own log-likelihood, normalized by length.
2. **Length-invariant**: Dividing by $|y|$ prevents the model from gaming reward by being verbose.

The SimPO loss adds a **target reward margin** $\gamma$:

$$\mathcal{L}_{\text{SimPO}} = -\mathbb{E}\left[\log \sigma\left(\frac{\beta}{|y_w|} \log \pi_\theta(y_w|x) - \frac{\beta}{|y_l|} \log \pi_\theta(y_l|x) - \gamma\right)\right]$$

The gamma parameter ensures a minimum reward margin between chosen and rejected, preventing the loss from becoming too easy.

In [ ]:
def simpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    chosen_lengths: torch.Tensor,
    rejected_lengths: torch.Tensor,
    beta: float = 2.0,
    gamma: float = 0.5,
) -> tuple[torch.Tensor, dict]:
    """
    Compute the SimPO (Simple Preference Optimization) loss.
    
    Key differences from DPO:
    - No reference model needed (saves reference-model memory and compute; note the frozen reference is forward-only, so savings are well under 50% of training memory)
    - Uses length-normalized log-likelihoods as implicit reward
    - Adds target reward margin gamma
    
    Args:
        policy_chosen_logps: Sum of log probs for chosen under policy. Shape: (batch_size,)
        policy_rejected_logps: Sum of log probs for rejected under policy. Shape: (batch_size,)
        chosen_lengths: Number of response tokens in chosen. Shape: (batch_size,)
        rejected_lengths: Number of response tokens in rejected. Shape: (batch_size,)
        beta: Temperature parameter (typically larger than DPO, e.g., 2.0).
        gamma: Target reward margin. Ensures minimum gap between chosen and rejected.
    
    Returns:
        loss: Scalar SimPO loss.
        metrics: Dict with reward statistics.
    """
    # Length-normalized log-likelihoods = implicit rewards
    chosen_rewards = (beta / chosen_lengths.float()) * policy_chosen_logps  # (batch_size,)
    rejected_rewards = (beta / rejected_lengths.float()) * policy_rejected_logps  # (batch_size,)
    
    # Reward margin with target gamma
    logits = chosen_rewards - rejected_rewards - gamma
    
    # Standard log-sigmoid loss (same as DPO, but on different rewards)
    loss = -F.logsigmoid(logits).mean()
    
    metrics = {
        'chosen_reward': chosen_rewards.mean().item(),
        'rejected_reward': rejected_rewards.mean().item(),
        'reward_margin': (chosen_rewards - rejected_rewards).mean().item(),
        'effective_margin_after_gamma': logits.mean().item(),
    }
    
    return loss, metrics


print("SimPO loss function defined.")

In [ ]:
# Test SimPO: demonstrating length normalization

print("=" * 60)
print("SimPO: Demonstrating length normalization")
print("=" * 60)

# Scenario 1: chosen and rejected have same length
print("\nScenario 1: Same length responses")
loss1, m1 = simpo_loss(
    policy_chosen_logps=torch.tensor([-10.0]),    # sum of log-probs
    policy_rejected_logps=torch.tensor([-15.0]),
    chosen_lengths=torch.tensor([20]),
    rejected_lengths=torch.tensor([20]),
    beta=2.0, gamma=0.5,
)
print(f"Loss: {loss1.item():.4f}")
print(f"Chosen avg reward: {m1['chosen_reward']:.4f}, Rejected avg reward: {m1['rejected_reward']:.4f}")

# Scenario 2: rejected is much longer (would be exploited by DPO)
print("\nScenario 2: Rejected is 3x longer (length exploitation test)")
loss2, m2 = simpo_loss(
    policy_chosen_logps=torch.tensor([-10.0]),    # 20 tokens, -0.5 avg
    policy_rejected_logps=torch.tensor([-30.0]),  # 60 tokens, -0.5 avg (SAME per-token quality!)
    chosen_lengths=torch.tensor([20]),
    rejected_lengths=torch.tensor([60]),
    beta=2.0, gamma=0.5,
)
print(f"Loss: {loss2.item():.4f}")
print(f"Chosen avg reward: {m2['chosen_reward']:.4f}, Rejected avg reward: {m2['rejected_reward']:.4f}")
print("Note: Length normalization makes rewards comparable despite different lengths!")

# Scenario 3: What DPO would see (no length normalization)
print("\nScenario 3: Without length normalization (DPO-style)")
print(f"DPO chosen sum-logps: -10.0, rejected sum-logps: -30.0")
print(f"DPO would see a large margin simply because rejected is longer.")
print(f"SimPO normalizes: chosen avg: {-10.0/20:.2f}, rejected avg: {-30.0/60:.2f} (equal, as they should be)")

In [ ]:
# Visualize the effect of gamma (target reward margin)

gammas = np.linspace(0, 3.0, 50)
losses_by_gamma = []

for g in gammas:
    l, _ = simpo_loss(
        policy_chosen_logps=torch.tensor([-10.0]),
        policy_rejected_logps=torch.tensor([-15.0]),
        chosen_lengths=torch.tensor([20]),
        rejected_lengths=torch.tensor([20]),
        beta=2.0, gamma=g,
    )
    losses_by_gamma.append(l.item())

plt.figure(figsize=(8, 4))
plt.plot(gammas, losses_by_gamma, linewidth=2)
plt.xlabel('Gamma (target reward margin)', fontsize=12)
plt.ylabel('SimPO Loss', fontsize=12)
plt.title('SimPO Loss vs Target Margin (gamma)', fontsize=14)
plt.axvline(x=0.5, color='r', linestyle='--', alpha=0.5, label='gamma=0.5 (typical)')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("Higher gamma = harder to satisfy the target margin = higher loss = stronger training signal.")
print("gamma=0: same as DPO without reference model (just length-normalized).")
print("gamma too high: loss stays high, model struggles to meet the margin requirement.")

### Why SimPO Matters

1. **Memory and compute savings**: No reference model. The frozen reference is forward-only (no gradients or optimizer states), so dropping it saves well under 50% of training memory -- the bigger wins are the saved forward passes and engineering simplicity.
2. **Length-robust**: Average log-likelihood prevents length exploitation.
3. **Simpler implementation**: Fewer models to manage, no synchronization.
4. **Competitive performance**: SimPO matches or beats DPO on AlpacaEval and MT-Bench.

### When SimPO Breaks

- Without a reference model, the policy can drift further from the SFT baseline (no explicit KL anchor).
- The gamma parameter requires tuning for each task.
- Average log-likelihood may not be the right reward signal for all tasks (e.g., code generation where every token matters equally).

---
## 7. ORPO: Odds Ratio Preference Optimization

**Paper**: Hong et al. (2024), "ORPO: Monolithic Preference Optimization without Reference Model"  
**Link**: [https://arxiv.org/abs/2403.07691](https://arxiv.org/abs/2403.07691)

### The Problem ORPO Solves

The standard alignment pipeline has **two separate stages**:
1. **SFT**: Train the model to produce good outputs (supervised fine-tuning)
2. **Alignment**: Train the model to prefer good outputs over bad ones (DPO/RLHF)

ORPO asks: **Can we do both in one stage?**

### The Key Insight: Odds Ratio as Preference Signal

ORPO adds a preference-aware penalty term to the standard SFT (cross-entropy) loss. The penalty uses the **odds ratio** between chosen and rejected responses:

$$\text{odds}(y|x) = \frac{P(y|x)}{1 - P(y|x)}$$

The odds ratio between chosen and rejected:

$$\text{OR}(y_w, y_l | x) = \frac{\text{odds}(y_w|x)}{\text{odds}(y_l|x)}$$

The ORPO loss:

$$\mathcal{L}_{\text{ORPO}} = \mathcal{L}_{\text{SFT}}(y_w) + \lambda \cdot \mathcal{L}_{\text{OR}}$$

where:
$$\mathcal{L}_{\text{OR}} = -\log \sigma\left(\log \frac{\text{odds}(y_w|x)}{\text{odds}(y_l|x)}\right)$$

In practice, for language models where per-token probabilities are small, the odds ratio simplifies to approximately the log-probability ratio.

In [ ]:
def orpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    sft_loss: torch.Tensor,
    lambda_orpo: float = 1.0,
) -> tuple[torch.Tensor, dict]:
    """
    Compute the ORPO (Odds Ratio Preference Optimization) loss.
    
    Key innovation: combines SFT and alignment in ONE loss.
    No reference model needed. No separate alignment stage.
    
    Args:
        policy_chosen_logps: Average log probs of chosen response. Shape: (batch_size,)
        policy_rejected_logps: Average log probs of rejected response. Shape: (batch_size,)
        sft_loss: SFT (cross-entropy) loss on chosen responses. Scalar.
        lambda_orpo: Weight for the odds-ratio penalty.
    
    Returns:
        loss: Scalar ORPO loss (SFT + lambda * OR penalty).
        metrics: Dict with component losses.
    """
    # Convert log-probs to probabilities for odds computation
    # For language models, P(y|x) = exp(avg_logp) is the average token probability
    # We work in log-space for numerical stability
    
    # Log-odds: log(P/(1-P)) = logP - log(1-P)
    # For small P (typical in LMs), log(1-P) ~ -P ~ -exp(logP)
    # But we use the exact computation:
    chosen_log_odds = policy_chosen_logps - torch.log1p(-torch.exp(policy_chosen_logps).clamp(max=1.0 - 1e-7))
    rejected_log_odds = policy_rejected_logps - torch.log1p(-torch.exp(policy_rejected_logps).clamp(max=1.0 - 1e-7))
    
    # Log odds ratio = log(odds_chosen / odds_rejected) = log_odds_chosen - log_odds_rejected
    log_odds_ratio = chosen_log_odds - rejected_log_odds  # (batch_size,)
    
    # Odds ratio loss: negative log-sigmoid of log odds ratio
    or_loss = -F.logsigmoid(log_odds_ratio).mean()
    
    # Combined loss
    loss = sft_loss + lambda_orpo * or_loss
    
    metrics = {
        'sft_loss': sft_loss.item(),
        'or_loss': or_loss.item(),
        'log_odds_ratio': log_odds_ratio.mean().item(),
        'total_loss': loss.item(),
    }
    
    return loss, metrics


print("ORPO loss function defined.")

In [ ]:
# Test ORPO loss

print("=" * 60)
print("TEST: ORPO loss behavior")
print("=" * 60)

# Note: for ORPO, log-probs should be average per-token log-probs
# (since we're computing odds on average token probabilities)

# Scenario: Model correctly prefers chosen
sft_loss = torch.tensor(2.5)  # Typical SFT loss

loss1, m1 = orpo_loss(
    policy_chosen_logps=torch.tensor([-0.5]),     # avg logp: high prob for chosen
    policy_rejected_logps=torch.tensor([-2.0]),   # avg logp: low prob for rejected
    sft_loss=sft_loss,
    lambda_orpo=1.0,
)
print(f"\nCorrect preference (chosen > rejected):")
print(f"Total loss: {m1['total_loss']:.4f} = SFT({m1['sft_loss']:.4f}) + OR({m1['or_loss']:.4f})")
print(f"Log odds ratio: {m1['log_odds_ratio']:.4f} (positive = correct)")

# Scenario: Model wrongly prefers rejected
loss2, m2 = orpo_loss(
    policy_chosen_logps=torch.tensor([-2.0]),     # low prob for chosen
    policy_rejected_logps=torch.tensor([-0.5]),   # high prob for rejected
    sft_loss=sft_loss,
    lambda_orpo=1.0,
)
print(f"\nWrong preference (rejected > chosen):")
print(f"Total loss: {m2['total_loss']:.4f} = SFT({m2['sft_loss']:.4f}) + OR({m2['or_loss']:.4f})")
print(f"Log odds ratio: {m2['log_odds_ratio']:.4f} (negative = wrong)")

# Effect of lambda_orpo
print("\n" + "=" * 60)
print("Effect of lambda_orpo (alignment strength)")
print("=" * 60)

for lam in [0.0, 0.1, 0.5, 1.0, 5.0, 10.0]:
    l, m = orpo_loss(
        policy_chosen_logps=torch.tensor([-0.5]),
        policy_rejected_logps=torch.tensor([-2.0]),
        sft_loss=sft_loss,
        lambda_orpo=lam,
    )
    print(f"lambda={lam:5.1f} -> total={m['total_loss']:.4f} = SFT({m['sft_loss']:.4f}) + {lam}*OR({m['or_loss']:.4f})")

print("\nlambda=0: pure SFT (no alignment)")
print("lambda=1: balanced SFT + alignment")
print("lambda=10: alignment-dominated (may hurt language quality)")

### Why ORPO Matters

1. **Single stage**: No separate SFT then DPO pipeline. Train once, get both language quality and alignment.
2. **No reference model**: Like SimPO, saves the reference model's memory and forward passes (well under 50% of training memory, since the frozen reference holds no gradients or optimizer states).
3. **Simpler pipeline**: Fewer moving parts, fewer things to go wrong.
4. **Efficiency**: One training run instead of two.

### When ORPO Breaks

- The lambda_orpo parameter controls the trade-off between SFT and alignment. If it's too high, the model may sacrifice language quality for preference matching.
- Starting from a pretrained (not SFT'd) model may require more careful tuning.
- The odds ratio approximation can be less accurate for very short sequences.
- Mixing SFT and alignment objectives can create conflicting gradients.

---
## 8. Comprehensive Comparison Table

### All Methods Side by Side

| Dimension | DPO | KTO | IPO | SimPO | ORPO |
|-----------|-----|-----|-----|-------|------|
| **Data requirement** | Paired preferences | Unpaired (good/bad labels) | Paired preferences | Paired preferences | Paired preferences |
| **Reference model** | Yes | Yes | Yes | **No** | **No** |
| **Models in memory** | 2 | 2 | 2 | **1** | **1** |
| **Training stages** | 2 (SFT + DPO) | 2 (SFT + KTO) | 2 (SFT + IPO) | 2 (SFT + SimPO) | **1 (combined)** |
| **Loss function** | log-sigmoid on log-ratio margin | Prospect theory (asymmetric sigmoid) | Squared deviation from target margin | log-sigmoid on length-normalized margin | SFT + log-sigmoid on odds ratio |
| **Key hyperparameter** | beta | beta + lambda_D/lambda_U | tau | beta + gamma | lambda_orpo |
| **Length handling** | Unnormalized (vulnerable) | Unnormalized | Unnormalized | **Length-normalized** | Depends on implementation |
| **Overfitting risk** | Moderate (sigmoid saturation) | Low | **Low (squared loss)** | Moderate | Low |
| **Implementation complexity** | Low | Low | Low | Very low | Low |
| **Typical use case** | General alignment | Limited to binary feedback only | Noisy preferences | Memory-constrained settings | Want single-stage training |

### Decision Tree: When to Use Which

```
Do you have paired preference data?
|
+-- NO --> KTO (works with unpaired good/bad labels)
|
+-- YES --> Is GPU memory tight?
    |
    +-- YES --> Do you also need SFT?
    |   |
    |   +-- YES --> ORPO (single stage, no reference model)
    |   |
    |   +-- NO --> SimPO (no reference model, length-normalized)
    |
    +-- NO --> Is your preference data noisy?
        |
        +-- YES --> IPO (robust to overfitting) or DPO with label smoothing
        |
        +-- NO --> DPO (well-tested, well-understood)
```

**Insider Tip:** At Anthropic, the alignment method used in production isn't any single paper's algorithm -- it's a combination of techniques, iterated over many rounds, with heavy emphasis on data quality. Don't memorize one method; understand the design space and trade-offs. In an interview, the candidate who says "I'd start with DPO for simplicity, then iterate with on-policy data, and invest most of my effort in data curation" will beat the candidate who memorizes every formula but can't reason about when to use what.

In [ ]:
# Grand comparison: all loss functions on the same synthetic data

# Sweep: vary the policy's preference margin
margins = np.linspace(-4, 8, 100)

all_losses = {'DPO': [], 'KTO (desirable)': [], 'IPO': [], 'SimPO': [], 'ORPO': []}

ref_lp = -3.0

for m in margins:
    chosen_lp = ref_lp + m / 2
    rejected_lp = ref_lp - m / 2
    
    # DPO
    dl = dpo_loss(
        torch.tensor([chosen_lp]), torch.tensor([rejected_lp]),
        torch.tensor([ref_lp]), torch.tensor([ref_lp]), beta=0.1
    )
    all_losses['DPO'].append(dl.item())
    
    # KTO (just the desirable component for comparison)
    # NOTE: with a batch of ONE desirable example, z_ref equals the example's
    # own scaled logratio, so the loss would be a constant 0.5 regardless of
    # the margin. Include a small fixed batch of undesirable examples so z_ref
    # is a meaningful reference point and the KTO curve varies with the margin.
    kto_policy_lps = torch.tensor([chosen_lp, ref_lp - 1.0, ref_lp - 2.0, ref_lp + 0.5])
    kto_ref_lps = torch.full((4,), ref_lp)
    kto_labels = torch.tensor([True, False, False, False])
    _, kto_metrics = kto_loss(kto_policy_lps, kto_ref_lps, kto_labels, beta=0.1)
    all_losses['KTO (desirable)'].append(kto_metrics['desirable_loss'])
    
    # IPO
    il, _ = ipo_loss(
        torch.tensor([chosen_lp]), torch.tensor([rejected_lp]),
        torch.tensor([ref_lp]), torch.tensor([ref_lp]), tau=0.5
    )
    all_losses['IPO'].append(il.item())
    
    # SimPO (using length=20 for both)
    sl, _ = simpo_loss(
        torch.tensor([chosen_lp * 20]),  # sum logps = avg * length
        torch.tensor([rejected_lp * 20]),
        torch.tensor([20]), torch.tensor([20]),
        beta=2.0, gamma=0.5
    )
    all_losses['SimPO'].append(sl.item())
    
    # ORPO
    # Use average log-probs clamped to prevent numerical issues
    avg_chosen = max(chosen_lp, -10.0)
    avg_rejected = max(rejected_lp, -10.0)
    ol, _ = orpo_loss(
        torch.tensor([avg_chosen]),
        torch.tensor([avg_rejected]),
        sft_loss=torch.tensor(0.0),  # zero SFT for fair comparison
        lambda_orpo=1.0,
    )
    all_losses['ORPO'].append(ol.item())

# Plot
plt.figure(figsize=(12, 6))
for name, losses in all_losses.items():
    plt.plot(margins, losses, label=name, linewidth=2)

plt.xlabel('Log-ratio margin (chosen - rejected)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('All Alignment Losses vs Preference Margin', fontsize=14)
plt.legend(fontsize=11)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
plt.ylim(0, 5)
plt.tight_layout()
plt.show()

print("Observations:")
print("- DPO/SimPO/ORPO: sigmoid-based losses that saturate for large margins (vanishing gradients)")
print("- IPO: squared loss that grows for very large margins (actively prevents overfitting)")
print("- KTO: different shape because it operates on individual examples relative to a")
print("  reference point z_ref (estimated here from a small fixed batch), not on pairs")
print("- All losses are minimized when the model correctly prefers chosen over rejected")

---
## 9. "Why Does This Work?" -- Deep Understanding

### Each Method's Key Assumption and When It Breaks

| Method | Key Assumption | When It Breaks |
|--------|---------------|----------------|
| **DPO** | Preferences follow Bradley-Terry model; offline data is representative | Noisy labels; distribution shift; length exploitation |
| **KTO** | Binary quality labels are reliable; prospect theory models human preferences well | Very noisy labels; imbalanced desirable/undesirable distributions |
| **IPO** | A fixed target margin is appropriate for all pairs | Some pairs have clear preferences and others are ambiguous; the squared loss is sensitive to outliers |
| **SimPO** | Average log-likelihood is a good reward signal; length normalization is sufficient | Tasks where length matters (e.g., detailed explanations); no KL anchor can lead to policy drift |
| **ORPO** | SFT and alignment can be learned jointly without conflicts | Conflicting gradients between SFT and alignment; lambda_orpo tuning is task-sensitive |

### The Trend: Simpler Methods, Fewer Models, Less Memory

The field is clearly moving toward simplification:

1. **RLHF**: 4 models, 3 phases, complex RL infrastructure
2. **DPO**: 2 models, 2 phases, supervised learning
3. **SimPO/ORPO**: 1 model, 1-2 phases, minimal infrastructure

This trend reflects a practical reality: **most teams don't need the theoretical optimality of RLHF**. The simpler methods work well enough for most applications, and their simplicity enables faster iteration.

### The Trade-Off Triangle

```
        Performance
           /\
          /  \
         /    \
        / RLHF \
       /   DPO   \
      / IPO  KTO   \
     / SimPO  ORPO   \
    /________________\
  Simplicity -------- Robustness
```

- **Performance ceiling**: RLHF > DPO > SimPO/ORPO (generally, with exceptions)
- **Simplicity**: ORPO > SimPO > DPO > RLHF
- **Robustness to noise**: IPO > KTO > DPO > SimPO (generally)

### Interview Perspective

At a frontier lab interview, you should be able to:
1. **Derive DPO from RLHF** (this is the most common question)
2. **Explain the motivation for each variant** (what DPO limitation does it address?)
3. **Write the loss function for any of these methods** (at least approximately)
4. **Discuss trade-offs** (when would you use DPO vs. SimPO vs. RLHF?)
5. **Know the failure modes** (what goes wrong with each method?)

---
## Interview Question Bank: Alignment Variants (KTO, IPO, SimPO, ORPO)

*These questions test whether you understand the alignment method landscape deeply enough to make real engineering decisions -- not just recite paper summaries. At the Senior/Principal level, you are expected to recommend the right method for a given situation and defend that choice under pressure.*

---

**Q1: "You have 100K examples of human-rated outputs (good/bad labels, NOT paired). Which alignment method do you use and why?"**

**What we're testing:** Practical method selection, knowledge of data requirements, ability to match constraints to algorithms.

**Good answer:**
- KTO (Kahneman-Tversky Optimization) -- specifically designed for unpaired binary feedback (desirable/undesirable labels)
- Does not require paired preferences (A > B). Works with independent "thumbs up / thumbs down" labels.
- Discusses the prospect theory motivation: humans are loss-averse, so the loss function weights undesirable examples more heavily than desirable ones (asymmetric treatment).
- Implementation: uses the same reference model setup as DPO, but the loss operates on individual examples rather than pairs.

**Great answer (Senior -> Principal level):**
- Quantifies the data advantage: "With 100K unpaired examples, I could construct at most ~50K pairs by random pairing, but that wastes information and introduces noise from arbitrary pairings. KTO uses all 100K examples directly."
- Discusses when KTO underperforms DPO: "If the good/bad labels are noisy (e.g., binary labels on a nuanced quality spectrum), KTO can struggle because it loses the relative ranking signal that paired preferences provide."
- Mentions the ratio sensitivity: KTO performance depends on the ratio of desirable to undesirable examples. The paper explicitly handles extreme class imbalance via the lambda_D/lambda_U weights (guidance: set lambda_D*n_D / (lambda_U*n_U) in [1, 4/3]), so skewed real-world data does not need to be resampled to 50/50.
- Connects to production reality: "Most real-world feedback is unpaired -- user thumbs up/down, flagged responses, implicit signals. KTO is often the most practical choice when you are building alignment from real user feedback."

**Red flag:** Suggests DPO and proposes "just pair them randomly." Does not know KTO exists. Cannot explain why unpaired data is fundamentally different from paired data.

**Follow-up:** "What if you also have 10K paired preferences? How would you combine them?"
- Expected: Multi-stage approach: train KTO on the 100K unpaired data first, then fine-tune with DPO on the 10K paired data. Or: train a reward model on the paired data, use it to re-label or filter the unpaired data, then train KTO. A great candidate discusses curriculum learning -- start with the noisier signal (unpaired), refine with the cleaner signal (paired).

---

**Q2: "Your team wants to minimize inference cost. The current pipeline uses DPO with a reference model. How do you eliminate the reference model?"**

**What we're testing:** Knowledge of reference-free methods, memory optimization, practical engineering trade-offs.

**Good answer:**
- SimPO: Reference-free, uses length-normalized average log probability as the implicit reward. The reward for a response is just the average log-prob under the policy, divided by response length. No reference model needed at training or inference.
- ORPO: Combines SFT and alignment into a single stage using an odds ratio penalty. No separate reference model, no separate SFT step.
- Both remove the reference model from training. Since the frozen reference is forward-only (no gradients or optimizer states), this saves well under 50% of training memory; the bigger savings are compute and engineering simplicity.

**Great answer (Senior -> Principal level):**
- Discusses the theoretical cost of removing the reference model: "The reference model acts as an anchor that prevents the policy from drifting too far. Without it, you risk reward hacking or mode collapse. SimPO mitigates this through length normalization (which acts as an implicit regularizer), while ORPO mitigates it through the odds ratio formulation."
- Compares SimPO vs ORPO on quality: "SimPO-vs-ORPO rankings vary by benchmark and setup -- neither consistently dominates. ORPO's distinct advantage is that it folds SFT and alignment into a single stage, eliminating a separate SFT pass; how much wall-clock time that saves depends on your pipeline."
- Notes the gamma parameter in SimPO: "The target reward margin (gamma) controls how much the model must prefer chosen over rejected. Too low and it underfits; too high and it overfits. This replaces the beta parameter's role in DPO."
- Mentions that in practice, many teams keep the reference model but use a frozen copy on cheaper hardware (e.g., CPU offloading with DeepSpeed). "Eliminating the reference model is not always worth the quality trade-off."

**Red flag:** Does not know any reference-free methods. Suggests "just remove the KL term from DPO" without understanding the consequences (degenerate solutions).

**Follow-up:** "Does removing the reference model hurt quality? How would you measure?"
- Expected: Run a controlled experiment -- same data, same base model, same compute budget. DPO with reference vs SimPO without. Measure: win rate (head-to-head with judge model), safety regression (harmbench), diversity (distinct-n, entropy), and length distribution. A great candidate notes that the answer depends on the domain: for safety-critical applications, the reference model's regularization is valuable; for general helpfulness, SimPO often matches or exceeds DPO.

---

**Q3: "You are choosing between DPO, KTO, SimPO, and ORPO for a new model. Walk me through your decision process."**

**What we're testing:** Systems thinking, ability to reason about multiple constraints simultaneously, leadership-level decision-making.

**Good answer:**
Lists the key factors: data format, compute budget, quality requirements, and provides a reasonable recommendation.

**Great answer (Principal level -- this is a decision-tree question):**

"I would build a decision tree based on four axes:

**Axis 1: Data availability**
- Paired preferences available? -> DPO family (DPO, SimPO, IPO)
- Only unpaired labels? -> KTO
- No preference data at all, only demonstrations? -> Start with SFT, then generate synthetic preferences

**Axis 2: Compute budget**
- Memory-constrained (single-GPU or small cluster)? -> SimPO (no reference model) or ORPO (no separate SFT stage)
- Compute-rich (large cluster)? -> DPO with iterative refinement, or GRPO for reasoning tasks

**Axis 3: Quality requirements**
- Maximum quality, cost is secondary? -> Iterative DPO with on-policy data refresh (Llama 3 recipe)
- Good quality, minimize training time? -> ORPO (single stage)
- Good quality, minimize memory? -> SimPO

**Axis 4: Task characteristics**
- Style/safety alignment? -> DPO or KTO (well-understood, stable)
- Reasoning capabilities? -> GRPO (skip this entire family, go to RL)
- Instruction following? -> Any of the above work; DPO with high-quality data is the default

Then I would validate with a small-scale experiment (1B model, 10K examples) before committing to the full training run."

**Red flag:** Picks one method without considering constraints. Says "always use DPO" or "the newest method is best." Cannot articulate trade-offs between any two methods.

---
## Production Implementation Notes: Alignment Variants

### When to Use What -- A Practitioner's Cheatsheet

| Scenario | Best Method | Why | Second Choice |
|----------|-------------|-----|---------------|
| Paired preferences, large compute budget | Iterative DPO | Gold standard, well-understood | SimPO (if memory-constrained) |
| Unpaired binary feedback (thumbs up/down) | KTO | Only method designed for this data format | Convert to pairs + DPO (lossy) |
| Single-GPU fine-tuning | SimPO | No reference model = meaningful memory/compute savings (well under 50% of training memory) | ORPO (also reference-free, single-stage) |
| Tight timeline, need alignment fast | ORPO | Combines SFT + alignment in one stage | SimPO (but needs SFT first) |
| Noisy/low-quality preference data | IPO | Squared loss resists overfitting to noise | DPO with high beta |
| Safety-critical application | DPO with strong reference | Reference model prevents dangerous drift | Add safety-specific KTO pass |

### Combining Methods in Production

Frontier labs rarely use a single method in isolation. The real recipe:

1. **SFT** on high-quality demonstrations (always the first step)
2. **DPO or SimPO** on paired preferences for general helpfulness alignment
3. **KTO** on unpaired safety feedback (flagged responses, red-team data) -- this is where KTO shines, because safety data is naturally unpaired
4. **Iterative refinement** with on-policy data generation and re-ranking

### Common Pitfalls

- **Switching methods mid-pipeline**: Each method makes different assumptions about the data distribution. If you train DPO and then switch to KTO, the KTO reference model should be the DPO-trained model, not the original SFT model.
- **Hyperparameter transfer**: Beta in DPO, lambda in KTO, gamma in SimPO, and lambda in ORPO are NOT interchangeable. Each controls a different trade-off. You must tune per-method.
- **Evaluation bias**: If you evaluate with a reward model trained on paired data, it will naturally favor DPO over KTO. Use human eval or diverse automated judges for fair comparison.

---
## How Alignment Variants Get Tested in Interviews

### What Interviewers Are Really Looking For

When an interviewer asks about KTO, SimPO, or ORPO, they are NOT testing whether you memorized each paper. They are testing:

1. **Can you reason about trade-offs?** -- Every method is a different point in a multi-dimensional trade-off space (data requirements, compute, memory, quality, stability). Can you navigate that space fluently?
2. **Do you know what matters in practice?** -- The theoretical differences between IPO and DPO are small. The practical differences (IPO resists noisy data better) are what matter when you are spending $500K on a training run.
3. **Can you make a recommendation and defend it?** -- "It depends" is not an answer at the Principal level. "Given these constraints, I would use X because Y, and here is how I would validate that choice" is.

### The "Alignment Zoo" Question Pattern

Nearly every frontier lab interview includes some version of: "We have [data type] and [compute budget]. Which alignment method do you use?"

The variables they change:
- Data: paired preferences, unpaired labels, demonstrations only, mixture
- Compute: single GPU, 8 GPUs, 1000 GPUs
- Timeline: 1 week, 1 month, 3 months
- Quality bar: "good enough for beta launch" vs "frontier model release"
- Safety requirements: consumer product vs research API vs enterprise

Practice building a decision tree for each combination. If you can answer any combination fluently in under 2 minutes, you are ready.

### Common Gotcha Questions

- "Why not just use the newest method?" -- Tests whether you understand that newer does not mean better. SimPO is newer than DPO but not always superior. The answer depends on your constraints.
- "Can you combine KTO and DPO in the same training run?" -- Tests whether you understand that these are different loss functions with different data requirements. You cannot naively combine them, but you can use multi-stage training.
- "IPO claims to fix DPO's overfitting problem. Does it?" -- Tests critical reading. IPO helps with noisy preferences but introduces its own issues (the squared loss can underfit on clean data).

---
## 10. Flashcard Summary (Anki-Ready Q&A)

| # | Question | Answer |
|---|----------|--------|
| 1 | What data does KTO need that DPO doesn't? | KTO works with UNPAIRED examples labeled as "desirable" or "undesirable". DPO requires PAIRED preferences (chosen vs. rejected for the same prompt). |
| 2 | What psychological theory inspires KTO? | Kahneman & Tversky's prospect theory: loss aversion (penalties for bad > rewards for good) and reference dependence (utility relative to a baseline). |
| 3 | What is z_ref in KTO? | A running estimate of the average KL divergence between policy and reference across the batch. It serves as the "reference point" in prospect theory. |
| 4 | What problem does IPO solve that DPO has? | DPO can overfit because the log-sigmoid loss saturates for large margins, causing vanishing gradients. IPO uses a squared loss that never saturates. |
| 5 | What is the IPO loss function? | L_IPO = (margin - 1/(2*tau))^2, where margin = log(pi(y_w)/pi_ref(y_w)) - log(pi(y_l)/pi_ref(y_l)). A squared deviation from a target margin. |
| 6 | What does SimPO remove that DPO requires? | The reference model. SimPO uses length-normalized average log-likelihood as the implicit reward instead of the log-ratio pi/pi_ref. |
| 7 | What is the SimPO reward? | r(x,y) = (1/\|y\|) * log pi(y\|x). The average per-token log-probability, which is length-invariant. |
| 8 | What is gamma in SimPO? | The target reward margin. It ensures a minimum gap between chosen and rejected rewards: L = -log sigma(r_chosen - r_rejected - gamma). |
| 9 | How much memory does SimPO save over DPO? | Well under 50%: the frozen reference holds no gradients or optimizer states, so dropping it removes only its forward-pass memory. The bigger savings are compute and simplicity. |
| 10 | What is ORPO's key innovation? | Combining SFT and alignment in a single training stage. The loss = SFT_loss + lambda * odds_ratio_loss. No separate alignment phase needed. |
| 11 | What is the odds ratio in ORPO? | OR = odds(chosen)/odds(rejected), where odds(y\|x) = P(y\|x)/(1-P(y\|x)). Measures how much more likely the chosen response is than the rejected one. |
| 12 | Which methods don't need a reference model? | SimPO and ORPO. Both use only the policy model, saving the reference model's memory and forward passes (well under 50% of training memory). |
| 13 | Which method works with unpaired data? | KTO. All others (DPO, IPO, SimPO, ORPO) require paired preferences. |
| 14 | How do you choose between these methods? | Decision tree: No pairs -> KTO. Memory tight + need SFT -> ORPO. Memory tight -> SimPO. Noisy data -> IPO. Default -> DPO. |
| 15 | What is the general trend in alignment methods? | Simpler is better: fewer models (4 -> 2 -> 1), fewer phases (3 -> 2 -> 1), less memory, simpler implementations. The trade-off is a potentially lower performance ceiling compared to full RLHF. |

---
## 11. Paper Guides

### Paper 1: KTO (Ethayarajh et al., 2024)

**Title**: "KTO: Model Alignment as Prospect Theoretic Optimization"  
**Link**: [https://arxiv.org/abs/2402.01306](https://arxiv.org/abs/2402.01306)

| Section | What to Focus On | Time |
|---------|-----------------|------|
| Abstract + Intro | Prospect theory motivation; the unpaired data advantage | 10 min |
| Sec 2: Background | Review of DPO and its limitations; prospect theory basics | 10 min |
| **Sec 3: KTO** | **Core contribution.** The loss function derivation. Understand the asymmetric treatment of desirable/undesirable. | 20 min |
| Sec 4: Experiments | KTO vs DPO on Mistral-7B. Note: KTO matches DPO with weaker supervision. | 10 min |
| Sec 5: Analysis | The role of lambda_D/lambda_U (class-imbalance weights; loss aversion comes from the value-function shape). Data efficiency analysis. | 10 min |

**Key takeaway**: You can get DPO-level alignment without paired preferences. This dramatically reduces data collection costs.

---

### Paper 2: IPO (Azar et al., 2023)

**Title**: "A General Theoretical Paradigm to Understand Learning from Human Feedback"  
**Link**: [https://arxiv.org/abs/2310.12036](https://arxiv.org/abs/2310.12036)

| Section | What to Focus On | Time |
|---------|-----------------|------|
| Abstract + Intro | The overfitting problem in DPO; general framework motivation | 10 min |
| **Sec 3: General Framework** | **Key section.** The Psi-PO framework that unifies RLHF, DPO, and IPO. | 20 min |
| Sec 4: IPO | The specific IPO loss and its properties. Why squared loss prevents overfitting. | 15 min |
| Sec 5: Experiments | IPO vs DPO on overfitting scenarios. | 10 min |

**Key takeaway**: DPO's deterministic reward-policy mapping can overfit. IPO's squared loss regularizes this.

---

### Paper 3: SimPO (Meng et al., 2024)

**Title**: "SimPO: Simple Preference Optimization with a Reference-Free Reward"  
**Link**: [https://arxiv.org/abs/2405.14734](https://arxiv.org/abs/2405.14734)

| Section | What to Focus On | Time |
|---------|-----------------|------|
| Abstract + Intro | The reference model problem; length exploitation in DPO | 10 min |
| **Sec 3: Method** | **Core contribution.** Length-normalized reward, gamma margin, no reference model. | 15 min |
| Sec 4: Experiments | AlpacaEval, MT-Bench, Arena-Hard results. SimPO matches/beats DPO. | 10 min |
| Sec 5: Analysis | Ablation on gamma, beta, and length normalization. Crucial for understanding. | 10 min |

**Key takeaway**: You can drop the reference model, normalize for length, and still get strong alignment, saving the reference model's memory and forward passes (well under 50% of training memory).

---

### Paper 4: ORPO (Hong et al., 2024)

**Title**: "ORPO: Monolithic Preference Optimization without Reference Model"  
**Link**: [https://arxiv.org/abs/2403.07691](https://arxiv.org/abs/2403.07691)

| Section | What to Focus On | Time |
|---------|-----------------|------|
| Abstract + Intro | The two-stage pipeline problem; monolithic training motivation | 10 min |
| **Sec 3: Method** | **Core contribution.** SFT + odds ratio in one loss. | 15 min |
| Sec 3.3: Why SFT alone is insufficient | Important: shows that SFT increases both chosen AND rejected probabilities. ORPO adds differentiation. | 10 min |
| Sec 4: Experiments | ORPO vs DPO vs SFT on several benchmarks. | 10 min |

**Key takeaway**: You can combine SFT and alignment in one pass, removing both the reference model and the separate alignment stage.